# Load Data

In [40]:
import duckdb

con = duckdb.connect(
    r"..\data-modeling\target\adp_assessment.duckdb",
    read_only=True,
)

df = con.sql("select * from fct_activity").df()
con.close()

df.head()

,activity_date,casino_id,player_id,game_id,game_type,total_wager_base,total_payout_base,revenue,total_spins,go_live_date,...,mech_188,mech_189,mech_190,mech_191,mech_192,mech_193,mech_194,mech_195,mech_196,mech_197
0,2026-06-22,208,2469448,5831,38,84809.572549,103926.225703,-19116.653154,127,2026-05-15,...,0,0,0,0,0,0,0,0,0,0
1,2026-06-22,208,2469448,4735,38,13674.039437,6815.279225,6858.760212,56,2026-05-29,...,0,0,0,0,0,0,0,0,0,0
2,2026-06-22,208,2469448,3451,38,420176.776104,442435.661296,-22258.885192,514,2026-06-01,...,1,0,0,0,0,0,0,0,0,0
3,2026-06-22,208,2469448,6624,38,197382.120274,224390.389760,-27008.269485,200,2026-05-18,...,1,0,0,0,0,0,0,0,0,0
4,2026-06-22,208,2469448,5369,38,53480.425717,36529.966072,16950.459645,186,2025-07-31,...,1,0,0,0,0,0,0,0,0,0


In [41]:
import numpy as np

revenue_vector = df["revenue"].to_numpy()

In [60]:
import warnings
from typing import List, Literal

warnings.filterwarnings("ignore")

import numpy as np
from clustermatic import AutoClusterizer
from sklearn.metrics import silhouette_samples


def cluster_themes_by_revenue_np(
    themes_matrix: np.ndarray,
    revenue_vector: np.ndarray,
    theme_names: List[str],
    target_type: Literal["negative", "positive"] = "negative",
) -> dict:
    """Cluster themes by their revenue performance in the requested direction."""
    revenue_mask = revenue_vector < 0 if target_type == "negative" else revenue_vector > 0
    n_themes = themes_matrix.shape[1]
    features = []

    for index in range(n_themes):
        column = themes_matrix[:, index]

        if np.std(column) == 0 or np.std(revenue_vector) == 0:
            correlation = 0.0
        else:
            correlation = np.corrcoef(column, revenue_vector)[0, 1]

        target_theme_mask = (column == 1) & revenue_mask
        target_revenue = revenue_vector[target_theme_mask]
        target_period_average = float(target_revenue.mean()) if target_revenue.size else 0.0
        target_impact = float(target_revenue.sum())
        features.append([correlation, target_period_average, target_impact])

    feature_matrix = np.asarray(features)
    clusterizer = AutoClusterizer(
        n_iterations=10,
        score_metric="silhouette",
        seed=42,
    )
    cluster_ids = clusterizer.fit_cluster(feature_matrix, save_model=False)
    selected_result = clusterizer.report.iloc[0]
    algorithm_name = str(selected_result["Algorithm"])
    best_score = float(selected_result["Best Score"])

    unique_labels, label_counts = np.unique(cluster_ids, return_counts=True)
    can_calculate_cluster_scores = (
        1 < len(unique_labels) < n_themes and np.all(label_counts >= 2)
    )
    if can_calculate_cluster_scores:
        sample_scores = silhouette_samples(clusterizer.preprocessed_data_, cluster_ids)
        cluster_scores = {
            cluster_id: float(sample_scores[cluster_ids == cluster_id].mean())
            for cluster_id in unique_labels
        }
    else:
        cluster_scores = {cluster_id: best_score for cluster_id in unique_labels}

    return {
        "names": theme_names,
        "algorithm": algorithm_name,
        "cluster_labels": [f"Cluster {cluster_id + 1}" for cluster_id in cluster_ids],
        "cluster_scores": [float(cluster_scores[cluster_id]) for cluster_id in cluster_ids],
        "revenue_corr": feature_matrix[:, 0].tolist(),
        "target_rev_avg": feature_matrix[:, 1].tolist(),
        "associated_revenue": (feature_matrix[:, 2] / 1000).tolist(),
        "target_type": target_type,
        "optimized_k": len(unique_labels),
        "silhouette_score": best_score,
    }

In [47]:
from typing import Literal

import numpy as np
import plotly.express as px


def plot_clusters(
    cluster_data: dict,
    target_type: Literal["negative", "positive"] = "negative",
):
    """Filter clusters by revenue direction and render an interactive strip chart."""
    target = cluster_data.get("target_type", target_type)
    unique_clusters = sorted(set(cluster_data["cluster_labels"]))
    filtered_clusters = []

    for cluster in unique_clusters:
        cluster_revenues = [
            revenue
            for revenue, label in zip(
                cluster_data["associated_revenue"], cluster_data["cluster_labels"]
            )
            if label == cluster
        ]
        if target == "negative" and np.mean(cluster_revenues) < 0:
            filtered_clusters.append(cluster)
        elif target == "positive" and np.mean(cluster_revenues) > 0:
            filtered_clusters.append(cluster)

    plot_data = {
        "Theme": [],
        "Cluster": [],
        "Cluster Details": [],
        "Revenue Impact ($K)": [],
        "Corr": [],
        "Target Period Avg": [],
        "Cluster Score": [],
    }

    for index, theme in enumerate(cluster_data["names"]):
        cluster_label = cluster_data["cluster_labels"][index]
        if cluster_label in filtered_clusters:
            cluster_score = cluster_data["cluster_scores"][index]
            plot_data["Theme"].append(theme)
            plot_data["Cluster"].append(cluster_label)
            plot_data["Cluster Details"].append(
                f"{cluster_data['algorithm']}<br>{cluster_label}<br>Mean silhouette: {cluster_score:.3f}"
            )
            plot_data["Revenue Impact ($K)"].append(
                cluster_data["associated_revenue"][index]
            )
            plot_data["Corr"].append(cluster_data["revenue_corr"][index])
            plot_data["Target Period Avg"].append(cluster_data["target_rev_avg"][index])
            plot_data["Cluster Score"].append(cluster_score)

    if not plot_data["Theme"]:
        print(f"No clusters were found with an overall {target} revenue average.")
        return None

    title_text = (
        "Interactive Automated Clustering: Themes Driving Negative Revenue"
        if target == "negative"
        else "Interactive Automated Clustering: Top Profit-Generating Themes"
    )

    fig = px.strip(
        plot_data,
        x="Cluster Details",
        y="Revenue Impact ($K)",
        color="Cluster",
        hover_name="Theme",
        hover_data=["Corr", "Target Period Avg", "Cluster Score"],
        title=title_text,
        labels={
            "Cluster Details": "Algorithm / Cluster / Mean Silhouette Score",
        },
    )
    fig.update_layout(
        xaxis_type="category",
        template="plotly_white",
        showlegend=False,
        title_x=0.5,
    )
    return fig

# Which Casinos Offer Similar Themes Leading to Profit ?

In [61]:
theme_columns = [col for col in df.columns if "theme_" in str(col).lower()]
themes_matrix = df[theme_columns].to_numpy()

clustered_dict = cluster_themes_by_revenue_np(
    themes_matrix=themes_matrix,
    revenue_vector=revenue_vector,
    theme_names=theme_columns,  # Passing the clean names for better plotting
    target_type="positive"
)

interactive_plot = plot_clusters(clustered_dict,target_type="positive")
interactive_plot

Optimizing KMeans
KMeans optimized in 1.43 seconds. Best score: 0.5117939433196427
Optimizing DBSCAN
DBSCAN optimized in 0.25 seconds. Best score: 0.3740388620627956
Optimizing MiniBatchKMeans
MiniBatchKMeans optimized in 1.74 seconds. Best score: 0.49708665123759244
Optimizing AgglomerativeClustering
AgglomerativeClustering optimized in 0.18 seconds. Best score: 0.6737780650753046
Optimizing OPTICS
OPTICS optimized in 1.65 seconds. Best score: 0.3681379304565981
Optimizing SpectralClustering
SpectralClustering optimized in 0.85 seconds. Best score: 0.6737780650753046


# Which Casinos Offer Similar Themes Leading to Loss ?

In [62]:
theme_columns = [col for col in df.columns if "theme_" in str(col).lower()]
themes_matrix = df[theme_columns].to_numpy()

clustered_dict = cluster_themes_by_revenue_np(
    themes_matrix=themes_matrix,
    revenue_vector=revenue_vector,
    theme_names=theme_columns,  # Passing the clean names for better plotting
    target_type="negative"
)

interactive_plot = plot_clusters(clustered_dict,target_type="negative")
interactive_plot

Optimizing KMeans
KMeans optimized in 1.51 seconds. Best score: 0.590550224059419
Optimizing DBSCAN
DBSCAN optimized in 0.23 seconds. Best score: 0.8192126623741613
Optimizing MiniBatchKMeans
MiniBatchKMeans optimized in 1.72 seconds. Best score: 0.5485246346917124
Optimizing AgglomerativeClustering
AgglomerativeClustering optimized in 0.25 seconds. Best score: 0.8192126623741613
Optimizing OPTICS
OPTICS optimized in 1.85 seconds. Best score: 0.49748942068458396
Optimizing SpectralClustering
SpectralClustering optimized in 0.85 seconds. Best score: 0.8192126623741613


# Which Casinos Offer Similar Mechanics Leading to Loss ?

In [64]:
mechanic_columns = [
    column for column in df.columns if str(column).lower().startswith("mech_")
]
mechanics_matrix = df[mechanic_columns].to_numpy()

clustered_dict = cluster_themes_by_revenue_np(
    themes_matrix=mechanics_matrix,
    revenue_vector=revenue_vector,
    theme_names=mechanic_columns,
    target_type="negative",
)

interactive_plot = plot_clusters(clustered_dict, target_type="negative")
interactive_plot

Optimizing KMeans
KMeans optimized in 1.50 seconds. Best score: 0.5576970068797634
Optimizing DBSCAN
DBSCAN optimized in 0.31 seconds. Best score: 0.45313209111122316
Optimizing MiniBatchKMeans
MiniBatchKMeans optimized in 1.91 seconds. Best score: 0.5505809148318228
Optimizing AgglomerativeClustering
AgglomerativeClustering optimized in 0.21 seconds. Best score: 0.5612176647103887
Optimizing OPTICS
OPTICS optimized in 2.73 seconds. Best score: 0.5236599008933481
Optimizing SpectralClustering
SpectralClustering optimized in 1.15 seconds. Best score: 0.675290183823145


# Which Casinos Offer Similar Mechanics Leading to Profit ?

In [65]:
mechanic_columns = [
    column for column in df.columns if str(column).lower().startswith("mech_")
]
mechanics_matrix = df[mechanic_columns].to_numpy()

clustered_dict = cluster_themes_by_revenue_np(
    themes_matrix=mechanics_matrix,
    revenue_vector=revenue_vector,
    theme_names=mechanic_columns,
    target_type="negative",
)

interactive_plot = plot_clusters(clustered_dict, target_type="negative")
interactive_plot

Optimizing KMeans
KMeans optimized in 1.44 seconds. Best score: 0.558482527604515
Optimizing DBSCAN
DBSCAN optimized in 0.28 seconds. Best score: 0.45313209111122316
Optimizing MiniBatchKMeans
MiniBatchKMeans optimized in 1.73 seconds. Best score: 0.55485807059511
Optimizing AgglomerativeClustering
AgglomerativeClustering optimized in 0.31 seconds. Best score: 0.5612176647103887
Optimizing OPTICS
OPTICS optimized in 2.61 seconds. Best score: 0.5236599008933481
Optimizing SpectralClustering
SpectralClustering optimized in 1.04 seconds. Best score: 0.675290183823145


# Game Similarity

## Approach
For every `game_id`, the similarity model combines two signals:

1. **Content similarity (75%)**: cosine similarity across game type, theme flags, and mechanic flags. This is the primary signal because these attributes are stable game descriptors.
2. **Behaviour similarity (25%)**: proximity of game-level activity profiles, using wager per spin, revenue per spin, revenue margin, active players, active casinos, and total spins.

The blended score ranges from 0 to 1, where higher values indicate closer games. Behaviour is given lower weight because the available activity history covers only two days; longer history would make it more reliable. This approach is interpretable and deterministic, but it does not capture unobserved player-preference relationships or game metadata not present in the source data.

In [68]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.preprocessing import StandardScaler


def build_game_similarity_index(
    activity: pd.DataFrame,
    content_weight: float = 0.75,
) -> dict:
    """Build a hybrid content-and-behaviour similarity index for every game."""
    if not 0 <= content_weight <= 1:
        raise ValueError("content_weight must be between 0 and 1.")

    theme_columns = [column for column in activity.columns if column.startswith("theme_")]
    mechanic_columns = [column for column in activity.columns if column.startswith("mech_")]
    content_columns = theme_columns + mechanic_columns

    game_content = activity.groupby("game_id", as_index=False).agg(
        game_type=("game_type", "first"),
        **{column: (column, "max") for column in content_columns},
    )
    game_behaviour = activity.groupby("game_id", as_index=False).agg(
        activity_rows=("game_id", "size"),
        active_days=("activity_date", "nunique"),
        active_casinos=("casino_id", "nunique"),
        active_players=("player_id", "nunique"),
        total_spins=("total_spins", "sum"),
        total_wager=("total_wager_base", "sum"),
        total_revenue=("revenue", "sum"),
    )
    profiles = game_content.merge(game_behaviour, on="game_id", validate="one_to_one")
    profiles["wager_per_spin"] = profiles["total_wager"] / profiles["total_spins"].clip(lower=1)
    profiles["revenue_per_spin"] = profiles["total_revenue"] / profiles["total_spins"].clip(lower=1)
    profiles["revenue_margin"] = profiles["total_revenue"] / profiles["total_wager"].clip(lower=1)

    game_type_features = pd.get_dummies(profiles["game_type"], prefix="game_type", dtype=float)
    content_features = np.hstack(
        [
            game_type_features.to_numpy(),
            profiles[content_columns].to_numpy(dtype=float),
        ]
    )
    content_similarity = cosine_similarity(content_features)

    behaviour_columns = [
        "wager_per_spin",
        "revenue_per_spin",
        "revenue_margin",
        "active_players",
        "active_casinos",
        "total_spins",
    ]
    behaviour_features = profiles[behaviour_columns].copy()
    for column in ["active_players", "active_casinos", "total_spins"]:
        behaviour_features[column] = np.log1p(behaviour_features[column])
    scaled_behaviour = StandardScaler().fit_transform(behaviour_features)
    behaviour_distance = euclidean_distances(scaled_behaviour)
    behaviour_similarity = 1 / (1 + behaviour_distance)

    similarity = (
        content_weight * content_similarity
        + (1 - content_weight) * behaviour_similarity
    )
    np.fill_diagonal(similarity, -np.inf)

    return {
        "profiles": profiles,
        "content_similarity": content_similarity,
        "behaviour_similarity": behaviour_similarity,
        "similarity": similarity,
        "content_weight": content_weight,
    }


def get_similar_games(
    game_id: int,
    similarity_index: dict,
    n: int = 5,
) -> pd.DataFrame:
    """Return the n most similar games to a supplied game_id."""
    profiles = similarity_index["profiles"].reset_index(drop=True)
    matches = np.flatnonzero(profiles["game_id"].to_numpy() == game_id)
    if not len(matches):
        raise KeyError(f"game_id {game_id} is not present in the similarity index.")

    source_index = matches[0]
    candidate_indices = np.argsort(similarity_index["similarity"][source_index])[::-1][:n]
    result = profiles.loc[candidate_indices, [
        "game_id", "game_type", "activity_rows", "active_players", "total_spins",
        "total_revenue", "revenue_margin",
    ]].copy()
    result.insert(0, "rank", range(1, len(result) + 1))
    result.insert(1, "source_game_id", game_id)
    result["similarity_score"] = similarity_index["similarity"][source_index, candidate_indices]
    result["content_similarity"] = similarity_index["content_similarity"][source_index, candidate_indices]
    result["behaviour_similarity"] = similarity_index["behaviour_similarity"][source_index, candidate_indices]
    return result.round({
        "similarity_score": 3,
        "content_similarity": 3,
        "behaviour_similarity": 3,
        "revenue_margin": 4,
    })


similarity_index = build_game_similarity_index(df)
profiles = similarity_index["profiles"]

# Worked examples: use the highest-activity game from up to three game types.
example_game_ids = (
    profiles.sort_values("activity_rows", ascending=False)
    .groupby("game_type", group_keys=False)
    .head(1)["game_id"]
    .head(3)
    .tolist()
)

for example_game_id in example_game_ids:
    print(f"Most similar games to {example_game_id}")
    display(get_similar_games(example_game_id, similarity_index, n=5))

# Sanity check: report how many returned neighbours share the source game type.
sanity_check = pd.concat(
    [get_similar_games(game_id, similarity_index, n=5) for game_id in example_game_ids],
    ignore_index=True,
)
source_types = profiles.set_index("game_id")["game_type"]
sanity_check["same_game_type"] = sanity_check.apply(
    lambda row: source_types[row["source_game_id"]] == row["game_type"],
    axis=1,
)
sanity_summary = sanity_check.groupby("source_game_id", as_index=False).agg(
    average_similarity=("similarity_score", "mean"),
    same_type_neighbours=("same_game_type", "sum"),
)
print("Sanity check: same-type neighbours out of the top five results")
display(sanity_summary)

Most similar games to 5369


,rank,source_game_id,game_id,game_type,activity_rows,active_players,total_spins,total_revenue,revenue_margin,similarity_score,content_similarity,behaviour_similarity
1192,1,5369,5377,38,4648,4211,454304,1.418712e+08,0.1591,0.806,0.947,0.381
1188,2,5369,5373,38,3552,3360,373609,2.015748e+07,0.0440,0.802,0.947,0.367
1196,3,5369,5381,38,228,211,21466,2.300536e+06,0.0928,0.765,0.947,0.217
1187,4,5369,5372,38,12570,11756,1414052,1.280488e+08,0.0416,0.685,0.736,0.533
1186,5,5369,5371,38,1947,1770,265285,5.267327e+07,0.0978,0.633,0.736,0.324


Most similar games to 5992


,rank,source_game_id,game_id,game_type,activity_rows,active_players,total_spins,total_revenue,revenue_margin,similarity_score,content_similarity,behaviour_similarity
1596,1,5992,5981,25,7476,6626,340440,5.085139e+06,0.0478,0.879,1.000,0.515
1603,2,5992,5994,25,444,404,3247,-1.736965e+06,-0.8068,0.810,1.000,0.240
1597,3,5992,5983,25,4,4,116,1.011521e+04,0.2130,0.788,1.000,0.152
1604,4,5992,5995,25,8,8,142,1.403473e+03,0.0740,0.667,0.833,0.166
1598,5,5992,5985,38,3,3,6,6.825964e+02,0.4245,0.657,0.833,0.128


Most similar games to 11904


,rank,source_game_id,game_id,game_type,activity_rows,active_players,total_spins,total_revenue,revenue_margin,similarity_score,content_similarity,behaviour_similarity
1772,1,11904,11905,23,261,255,3046,4.814164e+05,0.1761,0.833,1.000,0.330
1773,2,11904,12003,23,14,13,2129,-1.493450e+06,-1.0376,0.482,0.577,0.196
1675,3,11904,6375,38,4564,4314,640064,7.942330e+07,0.0661,0.346,0.365,0.288
157,4,11904,3016,38,23,22,3220,5.277609e+05,0.1436,0.320,0.348,0.235
153,5,11904,3010,38,28,28,1745,3.005447e+05,0.1899,0.319,0.348,0.230


Sanity check: same-type neighbours out of the top five results


,source_game_id,average_similarity,same_type_neighbours
0,5369,0.7382,5
1,5992,0.7602,4
2,11904,0.4600,2


## Worked Examples and Sanity Check

The examples above demonstrate the `get_similar_games(game_id, similarity_index, n=5)` capability. The results are broadly sensible: the top five results for game `5369` share its game type, and four of five results for game `5992` do as well. Their highest content-similarity scores are also high, indicating common themes, mechanics, and game type.

Game `11904` has a lower mean neighbour score and only two same-type neighbours in its top five. Treat this as a lower-confidence recommendation: it may reflect genuinely sparse matching content or weak behavioural evidence. This is an expected trade-off of the two-day activity window. A production version should incorporate a longer activity history, game names/provider metadata if available, and user-interaction signals such as co-play to strengthen the behavioural component.

The content-weight parameter can be tuned after stakeholder review. The current 75% content / 25% behaviour blend favours stable game attributes over short-term performance noise.

## Auto-Cluster Cohorts

Auto-clustering is used as an additional cohort signal. `AutoClusterizer` evaluates several clustering algorithms on game content and behaviour features, selects the highest-silhouette candidate, and assigns each game to its chosen cohort. Games in the same cohort receive a small 10% score boost; content and behaviour remain the dominant signals so recommendations still rank individual nearest neighbours rather than merely listing every game in a cluster.

In [ ]:
from clustermatic import AutoClusterizer


def add_auto_cluster_cohorts(
    similarity_index: dict,
    cohort_weight: float = 0.10,
) -> dict:
    """Add an automatically selected game cohort as a small similarity signal."""
    if not 0 <= cohort_weight <= 1:
        raise ValueError("cohort_weight must be between 0 and 1.")

    profiles = similarity_index["profiles"].copy()
    content_columns = [
        column
        for column in profiles.columns
        if column.startswith("theme_") or column.startswith("mech_")
    ]
    behaviour_columns = [
        "wager_per_spin",
        "revenue_per_spin",
        "revenue_margin",
        "active_players",
        "active_casinos",
        "total_spins",
    ]
    game_type_features = pd.get_dummies(
        profiles["game_type"],
        prefix="game_type",
        dtype=float,
    )
    auto_cluster_features = np.hstack(
        [
            game_type_features.to_numpy(),
            profiles[content_columns].to_numpy(dtype=float),
            profiles[behaviour_columns].to_numpy(dtype=float),
        ]
    )

    clusterizer = AutoClusterizer(
        n_iterations=10,
        score_metric="silhouette",
        seed=42,
    )
    cluster_ids = clusterizer.fit_cluster(auto_cluster_features, save_model=False)
    selected_result = clusterizer.report.iloc[0]
    algorithm = str(selected_result["Algorithm"])
    score = float(selected_result["Best Score"])

    profiles["auto_cluster"] = [
        f"Cluster {cluster_id + 1}" if cluster_id >= 0 else "Noise"
        for cluster_id in cluster_ids
    ]
    cohort_similarity = (cluster_ids[:, None] == cluster_ids[None, :]).astype(float)
    cohort_similarity[cluster_ids == -1, :] = 0
    cohort_similarity[:, cluster_ids == -1] = 0

    similarity_index["profiles"] = profiles
    similarity_index["auto_cluster_algorithm"] = algorithm
    similarity_index["auto_cluster_score"] = score
    similarity_index["similarity"] = (
        (1 - cohort_weight) * similarity_index["similarity"]
        + cohort_weight * cohort_similarity
    )
    np.fill_diagonal(similarity_index["similarity"], -np.inf)
    return similarity_index


_base_get_similar_games = get_similar_games


def get_similar_games(
    game_id: int,
    similarity_index: dict,
    n: int = 5,
) -> pd.DataFrame:
    """Return nearest games and their automatically selected cohort."""
    result = _base_get_similar_games(game_id, similarity_index, n)
    clusters = similarity_index["profiles"].set_index("game_id")["auto_cluster"]
    source_cluster = clusters[game_id]
    result.insert(2, "source_auto_cluster", source_cluster)
    result["candidate_auto_cluster"] = result["game_id"].map(clusters)
    result["same_auto_cluster"] = result["candidate_auto_cluster"] == source_cluster
    return result


similarity_index = add_auto_cluster_cohorts(similarity_index)
print(
    "Auto-clustering selected "
    f"{similarity_index['auto_cluster_algorithm']} "
    f"(silhouette score: {similarity_index['auto_cluster_score']:.3f})."
)

for example_game_id in example_game_ids:
    print(f"Most similar games to {example_game_id} with auto-cluster cohort")
    display(get_similar_games(example_game_id, similarity_index, n=5))

Optimizing KMeans
KMeans optimized in 4.24 seconds. Best score: 0.6468002117839537
Optimizing DBSCAN
DBSCAN optimized in 2.47 seconds. Best score: -0.10081248423232045
Optimizing MiniBatchKMeans
MiniBatchKMeans optimized in 3.56 seconds. Best score: 0.18545674087163672
Optimizing AgglomerativeClustering
AgglomerativeClustering optimized in 6.85 seconds. Best score: 0.6940668315721831
Optimizing OPTICS
OPTICS optimized in 175.89 seconds. Best score: 0.4979932821547849
Optimizing SpectralClustering
